# 1. outputs(shifted right) & output embedding


In [16]:
import math
import torch
import torch.nn as nn

# ===== R1：TokenEmbedding =====
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        # TODO R1.1: 建一个 nn.Embedding 查表层，记下 d_model
        self.emb = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
    def forward(self, ids):                      # ids: [B, n]
        # TODO R1.2: 查表，再乘 √d_model → [B, n, d]
        return self.emb(ids) * math.sqrt(self.d_model)

# ===== R2：PositionalEncoding =====
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        # TODO R2.1: pos，位置编号 0..max_len-1，形状 [max_len, 1]
        pos = torch.arange(0, max_len).unsqueeze(1)
        # TODO R2.2: div = 10000^(2i/d)，i 取偶数下标，形状 [d/2]
        two_i = torch.arange(0, d_model, 2)
        div = 10000 ** (two_i / d_model)
        # TODO R2.3: 偶数列放 sin(pos/div)，奇数列放 cos(pos/div)
        angle = pos / div
        pe[:, 0::2] = torch.sin(angle)
        pe[:, 1::2] = torch.cos(angle)
        # TODO R2.4: 用 register_buffer 存成 self.pe
        self.register_buffer("pe", pe)

    def forward(self, x):                        # x: [B, n, d]
        # TODO R2.5: 取前 n 个位置的 pe 加到 x 上
        n = x.size(1)
        return x + self.pe[:n]

# 自测
emb = TokenEmbedding(10, 8)
pe = PositionalEncoding(8)
ids = torch.tensor([[1, 5, 8, 9]])
print(pe(emb(ids)).shape)    # 应该是 [1, 4, 8]

torch.Size([1, 4, 8])


# 2. Masked Multi-Head Attention

In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        # TODO D2.1: 存 h 和 d_k，建 W_q、W_k、W_v、W_o 四个 Linear(d_model, d_model)
        self.h = h
        self.d_k = d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        

    def forward(self, x, causal=False):          # x: [B, n, d]
        B, n, d = x.shape
        # TODO D2.2: 算 Q、K、V，并拆成多头 → [B, h, n, d_k]
        #   提示：Linear → view(B, n, h, d_k) → transpose(1, 2)
        Q = self.W_q(x).view(B, n, self.h, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, n, self.h, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, n, self.h, self.d_k).transpose(1, 2)
        # TODO D2.3: 打分 score = Q·Kᵀ / √d_k → [B, h, n, n]
        score = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if causal:
            # TODO D2.4: 造下三角 mask [n, n]（对角线及以下为 1）
            mask = torch.tril(torch.ones(n, n))
            # TODO D2.5: mask == 0 的位置填 -inf
            score = score.masked_fill(mask == 0, float('-inf'))
        # TODO D2.6: softmax → 乘 V → 合并多头 [B, n, d] → W_o
        weight = torch.softmax(score, dim = -1)
        out = weight @ V
        out = out.transpose(1, 2).reshape(B, n, d)
        return self.W_o(out)

# 自测
x = torch.randn(1, 4, 8)
mha = MultiHeadAttention(8, 2)
print(mha(x, causal=True).shape)    # [1, 4, 8]

torch.Size([1, 4, 8])


# 3. Cross-Attention

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        # TODO D2.1: 存 h 和 d_k，建 W_q、W_k、W_v、W_o 四个 Linear(d_model, d_model)
        self.h = h
        self.d_k = d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, kv=None, causal=False):    # x: [B, m, d]  kv: [B, n, d]
        # TODO D3.1: kv 为 None 时就是自注意力，让 kv = x
        if kv is None:
            kv = x
        B, m, d = x.shape
        n = kv.size(1)
        # TODO D3.2: Q 用 x 算 → [B, h, m, d_k]
        #            K、V 用 kv 算 → [B, h, n, d_k]
        Q = self.W_q(x).reshape(B, m, self.h, self.d_k).transpose(1, 2)
        K = self.W_k(kv).reshape(B, n, self.h, self.d_k).transpose(1, 2)
        V = self.W_v(kv).reshape(B, n, self.h, self.d_k).transpose(1, 2)
        # score → [B, h, m, n]，后面和 D2 一样
        score = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        # TODO D3.3: 检查一下 mask、合并多头那几行里的 n，哪些要改成 m
        if causal:
            mask = torch.tril(torch.ones(m, n))
            score = score.masked_fill(mask == 0, float('-inf'))

        weight = torch.softmax(score, dim=-1)
        att = weight @ V
        att = att.transpose(1, 2).reshape(B, m, d)
        return self.W_o(att)

# 自测
mha = MultiHeadAttention(8, 2)
dec = torch.randn(1, 4, 8)      # decoder 4 个词
enc = torch.randn(1, 6, 8)      # encoder 输出 6 个词
print(mha(dec, kv=enc).shape)   # [1, 4, 8]
print(mha(dec, causal=True).shape)  # 自注意力还能用：[1, 4, 8]

torch.Size([1, 4, 8])
torch.Size([1, 4, 8])


# 4. FFN

In [21]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # TODO A.1: 第一个 Linear，d_model → d_ff
        self.linear1 = nn.Linear(d_model, d_ff)
        # TODO A.2: 第二个 Linear，d_ff → d_model
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):                # x: [B, n, d]
        # TODO A.3: Linear1 → ReLU → Linear2
        l1 = self.linear1(x)
        l2 = torch.relu(l1)
        l3 = self.linear2(l2)
        return l3

ffn = FeedForward(8, 32)
x = torch.randn(1, 4, 8)
print(ffn(x).shape)                     # [1, 4, 8]

torch.Size([1, 4, 8])


# 5. ADD & Norm

In [23]:
class AddNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        # TODO B.1: 建一个 nn.LayerNorm(d_model)
        self.layer = nn.LayerNorm(d_model)
        

    def forward(self, x, sub_out):      # x: 子层的输入，sub_out: 子层的输出，都是 [B, n, d]
        # TODO B.2: 先相加（Add），再归一化（Norm）
        return self.layer(x + sub_out)

addnorm = AddNorm(8)
y = addnorm(x, ffn(x))
print(y.shape)                          # [1, 4, 8]
print(y[0, 0].mean(), y[0, 0].std())    # 应该接近 0 和 1

torch.Size([1, 4, 8])
tensor(-7.4506e-09, grad_fn=<MeanBackward0>) tensor(1.0690, grad_fn=<StdBackward0>)
